In [11]:
import pyterrier as pt

# Pyterrier starten
if not pt.started():
    pt.init()

C:\Users\lenna\AppData\Local\Temp\ipykernel_65316\1740418717.py:4: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


In [15]:
# Dokumentkorpus laden
import json
from pathlib import Path

docs_path = (
    Path.home() / ".ir_datasets" / "longeval-sci-2026"
    / "longeval_sci_09_11_2026_documents" / "data" / "processed"
    / "doc_collection_09032026_abstract_2" / "snapshot-3"
    / "longeval_sci_test-09-11_2026_abstract" / "documents"
)

def document_generator():
    for file in sorted(docs_path.glob("documents_*.jsonl")):
        with open(file, "r", encoding="utf-8") as f:
            for line in f:
                doc = json.loads(line)
                yield {
                    "docno": doc["id"],
                    "text": f"{doc.get('title', '')} {doc.get('abstract', '')}".strip()
                }

In [16]:
# Index-Pfad definieren
BASE_DIR = Path.cwd()
index_path = str(BASE_DIR / "Index_Longeval_SCI_snapshot3")

In [17]:
# Doumentgenerator definieren
# Cell 4: Index bauen — NUR EINMAL AUSFÜHREN, danach auskommentieren
from pyterrier import IterDictIndexer

indexer = IterDictIndexer(
    index_path=index_path,
    overwrite=True,
    meta={"docno": 100, "text": 20480}
)

indexer.index(document_generator())

16:26:44.377 [main] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (44860) - further warnings are suppressed
16:58:45.519 [main] WARN org.terrier.structures.indexing.Indexer -- Indexed 6278 empty documents


<org.terrier.querying.IndexRef at 0x2b417f5bdd0 jclass=org/terrier/querying/IndexRef jself=<LocalRef obj=0x-56b4fbb0 at 0x2b417f25690>>

In [18]:
# Index laden und prüfen
index = pt.IndexFactory.of(index_path)
print(index.getCollectionStatistics().toString())

17:07:03.378 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 1,3 GiB of memory would be required.
Number of documents: 1661900
Number of terms: 2255473
Number of postings: 169210872
Number of fields: 0
Number of tokens: 281960829
Field names: []
Positions:   false



In [19]:
# Cell 6: Test mit BM25
bm25 = pt.terrier.Retriever(index, wmodel="BM25", metadata=["docno", "text"])
bm25.search("ai ethics").head()

,qid,docid,docno,text,rank,score,query
0,1,1551961,311144260,Proposal of Tools for the Practical Developmen...,0,22.779049,ai ethics
1,1,460420,164965707,A Capability Approach to AI Ethics We propose ...,1,22.614836,ai ethics
2,1,1646038,313710650,Bridging AI ethics between communication and c...,2,22.531300,ai ethics
3,1,672894,291105645,The Impact of AI Ethicality on Clinical Decisi...,3,22.511719,ai ethics
4,1,1084849,301805229,Ethical Considerations in the AI Lifecycle for...,4,22.430254,ai ethics


In [20]:
import pandas as pd

queries_path = Path.home() / ".ir_datasets" / "longeval-sci-2026" / "longeval_adhoc-queries-snapshot-test.tsv"

df = pd.read_csv(queries_path, sep="\t")
print(df.shape)
print(df.head())
print(df.columns.tolist())